In [9]:
%pip install hmmlearn tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import polars as pl

parquet_path = r"C:\Users\wongb\Downloads\ratings-20260117-20260217.parquet"
df = pl.read_parquet(parquet_path)
df

noteId,ratedOnTweetId,raterParticipantId,noteAuthorParticipantId,postCreatedAt,noteCreatedAt,ratingCreatedAt,helpfulnessLevel,fromNotification
i64,i64,str,str,datetime[ms],datetime[ms],datetime[ms],str,bool
1357294499052085249,1357269755112148993,"""FE4BA8F630E66459C9F2EEBDDF69C4…","""FB8A5B50F04AAB7CCEA99B4094EAA1…",2021-02-04 10:08:21.128,2021-02-04 11:46:40.544,2026-02-01 14:36:13.314,"""NOT_HELPFUL""",false
1357297935407603713,1357269755112148993,"""FE4BA8F630E66459C9F2EEBDDF69C4…","""B9BEE8138A8A67538391BC79C1E9DF…",2021-02-04 10:08:21.128,2021-02-04 12:00:19.835,2026-02-01 14:36:04.345,"""NOT_HELPFUL""",false
1358604958673719296,1358464064322691078,"""456D1C422BA1A09690B986C8A1DBE7…","""FAB1117EE31D648BA7C11062F626AF…",2021-02-07 17:14:06.633,2021-02-08 02:33:58.465,2026-02-09 08:56:34.712,"""NOT_HELPFUL""",false
1358604958673719296,1358464064322691078,"""4B3821FC937ACD882F8BD4045DD281…","""FAB1117EE31D648BA7C11062F626AF…",2021-02-07 17:14:06.633,2021-02-08 02:33:58.465,2026-02-09 00:51:34.502,"""NOT_HELPFUL""",false
1358844963216326659,1358464064322691078,"""4B3821FC937ACD882F8BD4045DD281…","""605A405C64CC57A943D086CC80E5AC…",2021-02-07 17:14:06.633,2021-02-08 18:27:40.007,2026-02-09 00:51:18.076,"""NOT_HELPFUL""",false
…,…,…,…,…,…,…,…,…
1873623034814238798,1873417011294093499,"""C6C2E0DD1A03ADE2BC4C0F1870E01E…","""27AC2BCC9AF39C20C5ECC5D5A45F32…",2024-12-29 17:13:16.554,2024-12-30 06:51:56.390,2026-02-14 01:57:17.194,"""HELPFUL""",false
1873640928910692708,1873299604080607330,"""E26C94838DB911EDA747183982B930…","""7F16863BE3FB87264952B6B64F8082…",2024-12-29 09:26:44.493,2024-12-30 08:03:02.674,2026-02-14 20:22:12.656,"""NOT_HELPFUL""",false
1873642906235351150,1873299604080607330,"""E26C94838DB911EDA747183982B930…","""BE38D44A6D5F6828EE25798913696D…",2024-12-29 09:26:44.493,2024-12-30 08:10:54.106,2026-02-14 20:22:02.614,"""NOT_HELPFUL""",false


In [5]:
ratings_preprocessed_df = (
    df
    .with_columns(
        pl.col("ratingCreatedAt")
        .cast(pl.Datetime, strict=False)
        .alias("ratingCreatedAt_ts")
    )
    .sort(["raterParticipantId", "ratingCreatedAt_ts"])
    .with_columns(
        pl.col("ratingCreatedAt_ts")
        .diff()
        .over("raterParticipantId")
        .dt.total_seconds()
        .alias("delta_seconds")
    )
    .with_columns(
        pl.when(pl.col("delta_seconds") > 0)
        .then(pl.col("delta_seconds").log())
        .otherwise(None)
        .alias("log_delta_seconds")
    )
)

ratings_preprocessed_df

noteId,ratedOnTweetId,raterParticipantId,noteAuthorParticipantId,postCreatedAt,noteCreatedAt,ratingCreatedAt,helpfulnessLevel,fromNotification,ratingCreatedAt_ts,delta_seconds,log_delta_seconds
i64,i64,str,str,datetime[ms],datetime[ms],datetime[ms],str,bool,datetime[μs],i64,f64
2016534713742061788,2016246745739546987,"""00002C7FD6E0080A69D0AB879C3D9B…","""FD7979DE7F187DC14A4FDF8F557242…",2026-01-27 20:27:38.894,2026-01-28 15:31:55.816,2026-01-28 20:47:22.321,"""HELPFUL""",false,2026-01-28 20:47:22.321,null,null
2017641230612451495,2017578816177111423,"""00002C7FD6E0080A69D0AB879C3D9B…","""2CE75633DDBFF074EE77E7DC582D7B…",2026-01-31 12:40:49.235,2026-01-31 16:48:49.996,2026-01-31 17:21:23.365,"""HELPFUL""",false,2026-01-31 17:21:23.365,246841,12.4165
2018404224149717157,2018387895459983768,"""00002C7FD6E0080A69D0AB879C3D9B…","""54F0002E8A4890D64140C45F5D108C…",2026-02-02 18:15:48.768,2026-02-02 19:20:41.831,2026-02-02 22:44:01.605,"""HELPFUL""",false,2026-02-02 22:44:01.605,192158,12.166073
2012648760455671942,2012586343008755726,"""00003D7D222733AE9D37A25B930204…","""B5A7C39C8965D9D8AF5CC38B994AAC…",2026-01-17 18:02:30.876,2026-01-17 22:10:32.357,2026-01-18 15:41:27.835,"""NOT_HELPFUL""",false,2026-01-18 15:41:27.835,null,null
2012693431319990371,2012586343008755726,"""00003D7D222733AE9D37A25B930204…","""D174DF64F45A03029780F273B4335C…",2026-01-17 18:02:30.876,2026-01-18 01:08:02.720,2026-01-18 15:44:04.360,"""HELPFUL""",false,2026-01-18 15:44:04.360,156,5.049856
…,…,…,…,…,…,…,…,…,…,…,…
2020752418963669036,2020604630657302841,"""FFFFBBAB3C66ABB4DBC2A3B486C3C6…","""BA48B1306E38D71468370309229E18…",2026-02-08 21:04:19.624,2026-02-09 06:51:35.102,2026-02-09 08:02:06.384,"""NOT_HELPFUL""",false,2026-02-09 08:02:06.384,1041422,13.856098
2021916824426701208,2021784932818067904,"""FFFFBBAB3C66ABB4DBC2A3B486C3C6…","""E6ABEA01CD00795F7897A1245141FF…",2026-02-12 03:14:25.588,2026-02-12 11:58:30.999,2026-02-12 15:50:43.969,"""HELPFUL""",false,2026-02-12 15:50:43.969,287317,12.568341
2019488960465031197,2019467286298472468,"""FFFFC46B8555A97065DB39F7D600C8…","""8D8A300EC055D96FC685C886688785…",2026-02-05 17:44:55.598,2026-02-05 19:11:03.122,2026-02-06 08:34:11.398,"""HELPFUL""",false,2026-02-06 08:34:11.398,null,null


In [6]:
ratings_per_user = (
    ratings_preprocessed_df
    .group_by("raterParticipantId")
    .agg(pl.len().alias("ratings_count"))
)

summary_stats_df = ratings_preprocessed_df.select(
    pl.len().alias("total_ratings"),
    pl.col("raterParticipantId").n_unique().alias("total_users"),
    pl.lit(ratings_per_user["ratings_count"].mean()).alias("avg_ratings_per_user"),
    pl.col("delta_seconds").mean().alias("avg_time_gap_seconds"),
    pl.col("delta_seconds").median().alias("median_time_gap_seconds"),
    pl.col("log_delta_seconds").mean().alias("avg_log_time_gap"),
    pl.col("log_delta_seconds").median().alias("median_log_time_gap"),
)

summary_stats_df

total_ratings,total_users,avg_ratings_per_user,avg_time_gap_seconds,median_time_gap_seconds,avg_log_time_gap,median_log_time_gap
u32,u32,f64,f64,f64,f64,f64
6499635,484059,13.427361,83461.493586,2012.0,7.165669,7.607878


In [15]:
import numpy as np
import polars as pl
from hmmlearn.hmm import GaussianHMM
from tqdm.auto import tqdm

# -----------------------------
# Run configuration (edit these)
# -----------------------------
SAMPLE_MODE = True  # Set to True to use a random sample of raters for faster iteration during development
SAMPLE_RATERS = 10000
RANDOM_SEED = 42
HMM_MAX_ITER = 50

if "ratings_preprocessed_df" not in globals():
    if "df" not in globals():
        raise ValueError("`df` not found. Run Cell 2 first to load your parquet data.")

    ratings_preprocessed_df = (
        df
        .with_columns(
            pl.col("ratingCreatedAt")
            .cast(pl.Datetime, strict=False)
            .alias("ratingCreatedAt_ts")
        )
        .sort(["raterParticipantId", "ratingCreatedAt_ts"])
        .with_columns(
            pl.col("ratingCreatedAt_ts")
            .diff()
            .over("raterParticipantId")
            .dt.total_seconds()
            .alias("delta_seconds")
        )
        .with_columns(
            pl.when(pl.col("delta_seconds") > 0)
            .then(pl.col("delta_seconds").log())
            .otherwise(None)
            .alias("log_delta_seconds")
        )
    )

progress = tqdm(total=7, desc="Session labeling", unit="step")

working_df = ratings_preprocessed_df
progress.set_postfix_str("choose data scope")

if SAMPLE_MODE:
    unique_raters_df = working_df.select("raterParticipantId").unique()
    total_raters = unique_raters_df.height
    n_sample = min(SAMPLE_RATERS, total_raters)

    sampled_raters = unique_raters_df.sample(
        n=n_sample, seed=RANDOM_SEED, shuffle=True
    )["raterParticipantId"].to_list()

    working_df = working_df.filter(pl.col("raterParticipantId").is_in(sampled_raters))
    print(f"SAMPLE_MODE=True -> using {n_sample} raters out of {total_raters}")
else:
    print("SAMPLE_MODE=False -> using all raters")
progress.update(1)

progress.set_postfix_str("sort + row index")
hmm_input_df = (
    working_df
    .sort(["raterParticipantId", "ratingCreatedAt_ts"])
    .with_row_index("row_nr")
)
progress.update(1)

progress.set_postfix_str("build observations")
valid_obs_df = hmm_input_df.filter(pl.col("log_delta_seconds").is_not_null())
if valid_obs_df.height == 0:
    raise ValueError("No valid non-null log_delta_seconds observations found.")

X = valid_obs_df["log_delta_seconds"].to_numpy().reshape(-1, 1)
progress.update(1)

progress.set_postfix_str("compute sequence lengths")
lengths = (
    valid_obs_df
    .group_by("raterParticipantId", maintain_order=True)
    .agg(pl.len().alias("n"))["n"]
    .to_list()
)

if len(lengths) == 0:
    raise ValueError("No user sequences found for HMM fitting.")
progress.update(1)

progress.set_postfix_str("fit HMM")
hmm = GaussianHMM(
    n_components=2,
    covariance_type="diag",
    n_iter=HMM_MAX_ITER,
    random_state=RANDOM_SEED,
)
hmm.fit(X, lengths=lengths)
progress.update(1)

progress.set_postfix_str("decode states")
decoded_states = hmm.predict(X, lengths=lengths)
state_means = hmm.means_.flatten()
in_session_state = int(np.argmin(state_means))
in_session_binary = (decoded_states == in_session_state).astype(np.int8)
progress.update(1)

progress.set_postfix_str("map to events")
state_df = pl.DataFrame(
    {
        "row_nr": valid_obs_df["row_nr"],
        "in_session_prev_gap": in_session_binary,
    }
)

ratings_session_df = (
    hmm_input_df
    .join(state_df, on="row_nr", how="left")
    .with_columns(
        pl.col("in_session_prev_gap")
        .shift(-1)
        .over("raterParticipantId")
        .alias("in_session_from_next_gap")
    )
    .with_columns(
        pl.coalesce([
            pl.col("in_session_prev_gap"),
            pl.col("in_session_from_next_gap"),
            pl.lit(0),
        ])
        .cast(pl.Int8)
        .alias("in_session")
    )
    .drop(["row_nr", "in_session_prev_gap", "in_session_from_next_gap"])
)
progress.update(1)
progress.close()

ratings_session_df

Session labeling:   0%|          | 0/7 [00:00<?, ?step/s]

SAMPLE_MODE=True -> using 10000 raters out of 484059


noteId,ratedOnTweetId,raterParticipantId,noteAuthorParticipantId,postCreatedAt,noteCreatedAt,ratingCreatedAt,helpfulnessLevel,fromNotification,ratingCreatedAt_ts,delta_seconds,log_delta_seconds,in_session
i64,i64,str,str,datetime[ms],datetime[ms],datetime[ms],str,bool,datetime[μs],i64,f64,i8
2021235037639999515,2021189166235562050,"""0013D5D5BE8EE0FA30A2F8E9DE1C01…","""813EB394B10809EBA8D09BCFA8E2E1…",2026-02-10 11:47:03.766,2026-02-10 14:49:20.361,2026-02-12 16:04:12.952,"""HELPFUL""",false,2026-02-12 16:04:12.952,null,null,0
2013386276955636109,2013327558041854176,"""0019812D65D12B55A7730E57DB7753…","""61DB37456FFAC3ED56B69E7C84CF91…",2026-01-19 19:07:50.311,2026-01-19 23:01:09.991,2026-01-20 07:34:28.735,"""HELPFUL""",false,2026-01-20 07:34:28.735,null,null,1
2013368773424664771,2013327558041854176,"""0019812D65D12B55A7730E57DB7753…","""3DB2C88F21A0C5421B373B4CAAC2C5…",2026-01-19 19:07:50.311,2026-01-19 21:51:36.824,2026-01-20 07:35:24.254,"""NOT_HELPFUL""",false,2026-01-20 07:35:24.254,55,4.007333,1
2013410511325470833,2013327558041854176,"""0019812D65D12B55A7730E57DB7753…","""F14ECB41C233875132EBFAC0D72206…",2026-01-19 19:07:50.311,2026-01-20 00:37:27.916,2026-01-20 07:35:38.037,"""NOT_HELPFUL""",false,2026-01-20 07:35:38.037,13,2.564949,1
2013943920124674233,2013418606399193378,"""0019812D65D12B55A7730E57DB7753…","""321BEBF06B9C14296569E1A26D2EB5…",2026-01-20 01:09:37.931,2026-01-21 11:57:02.484,2026-01-22 05:36:48.946,"""NOT_HELPFUL""",false,2026-01-22 05:36:48.946,165670,12.017753,0
…,…,…,…,…,…,…,…,…,…,…,…,…
2013236528898625683,2013095097915543746,"""FFFD98FC04D3E1615C8BF2617DA7EA…","""3AAE41D3F583557DAE9AACD4F4CFAC…",2026-01-19 03:44:07.498,2026-01-19 13:06:07.272,2026-01-19 16:15:35.891,"""HELPFUL""",false,2026-01-19 16:15:35.891,null,null,0
2014326838776955021,2014262176035242405,"""FFFD98FC04D3E1615C8BF2617DA7EA…","""018F93BCBCA9CF1F856F1A71C321E7…",2026-01-22 09:01:40.605,2026-01-22 13:18:37.405,2026-01-23 20:49:12.195,"""NOT_HELPFUL""",false,2026-01-23 20:49:12.195,362016,12.799444,0
2015903382687416347,2015888855799628186,"""FFFD98FC04D3E1615C8BF2617DA7EA…","""D20262A466B7EE36EF09892777C0FD…",2026-01-26 20:45:31.283,2026-01-26 21:43:14.763,2026-01-26 23:55:51.084,"""HELPFUL""",false,2026-01-26 23:55:51.084,270398,12.50765,0


In [ ]:
if "ratings_session_df" not in globals():
    raise ValueError("`ratings_session_df` not found. Run Cell 5 first.")

import polars as pl

# 1) Overall label distribution
label_counts_df = (
    ratings_session_df
    .group_by("in_session")
    .agg(pl.len().alias("n"))
    .sort("in_session")
)

total_rows = ratings_session_df.height
in_session_rows = label_counts_df.filter(pl.col("in_session") == 1).select(pl.col("n").sum()).item() or 0
in_session_rate = in_session_rows / total_rows if total_rows else 0.0

# 2) Per-user stats
per_user_df = (
    ratings_session_df
    .group_by("raterParticipantId")
    .agg(
        pl.len().alias("n_events"),
        pl.col("in_session").sum().alias("n_in_session"),
    )
    .with_columns(
        (pl.col("n_in_session") / pl.col("n_events")).alias("in_session_rate"),
        (pl.col("n_in_session") > 0).cast(pl.Int8).alias("has_any_in_session"),
        (pl.col("n_in_session") == pl.col("n_events")).cast(pl.Int8).alias("all_events_in_session"),
    )
    .sort("n_events", descending=True)
)

n_users = per_user_df.height
n_users_any_in = per_user_df.filter(pl.col("has_any_in_session") == 1).height
n_users_none_in = per_user_df.filter(pl.col("has_any_in_session") == 0).height
n_users_all_in = per_user_df.filter(pl.col("all_events_in_session") == 1).height

avg_user_in_rate = per_user_df.select(pl.col("in_session_rate").mean()).item()
median_user_in_rate = per_user_df.select(pl.col("in_session_rate").median()).item()

user_rate_quantiles_df = per_user_df.select(
    pl.col("in_session_rate").quantile(0.10).alias("p10"),
    pl.col("in_session_rate").quantile(0.25).alias("p25"),
    pl.col("in_session_rate").quantile(0.50).alias("p50"),
    pl.col("in_session_rate").quantile(0.75).alias("p75"),
    pl.col("in_session_rate").quantile(0.90).alias("p90"),
)

user_mix_summary_df = pl.DataFrame(
    {
        "metric": [
            "n_users",
            "n_users_any_in_session",
            "n_users_no_in_session",
            "n_users_all_in_session",
            "share_users_any_in_session",
            "share_users_no_in_session",
            "share_users_all_in_session",
            "avg_user_in_session_rate",
            "median_user_in_session_rate",
        ],
        "value": [
            float(n_users),
            float(n_users_any_in),
            float(n_users_none_in),
            float(n_users_all_in),
            (n_users_any_in / n_users) if n_users else 0.0,
            (n_users_none_in / n_users) if n_users else 0.0,
            (n_users_all_in / n_users) if n_users else 0.0,
            float(avg_user_in_rate) if avg_user_in_rate is not None else 0.0,
            float(median_user_in_rate) if median_user_in_rate is not None else 0.0,
        ],
    }
)

# 3) First-event fallback sanity: how many first events ended up as in_session=1
first_event_df = (
    ratings_session_df
    .sort(["raterParticipantId", "ratingCreatedAt_ts"])
    .group_by("raterParticipantId", maintain_order=True)
    .first()
    .select("raterParticipantId", "in_session")
)

first_event_summary_df = first_event_df.select(
    pl.len().alias("n_users"),
    pl.col("in_session").sum().alias("n_first_event_in_session"),
    (pl.col("in_session").sum() / pl.len()).alias("first_event_in_session_rate"),
)

print(f"Rows: {total_rows:,}")
print(f"In-session rows: {in_session_rows:,} ({in_session_rate:.2%})")

print("\nLabel counts:")
display(label_counts_df)

print("\nRater-level summary:")
display(user_mix_summary_df)

print("\nRater in-session-rate quantiles:")
display(user_rate_quantiles_df)

print("\nFirst-event sanity:")
display(first_event_summary_df)

print("\nTop 10 users by number of events:")
display(per_user_df.head(10))

Rows: 137,917
In-session rows: 53,496 (38.79%)

Label counts:


in_session,n
i8,u32
0,84421
1,53496



First-event sanity:


n_users,n_first_event_in_session,first_event_in_session_rate
u32,i64,f64
10000,2075,0.2075



Top 10 users by number of events:


raterParticipantId,n_events,n_in_session,in_session_rate
str,u32,i64,f64
"""1A3A2463BDFACCE31C3F2BFB58EA91…",1212,801,0.660891
"""049E0D5E73AF4A77B3E19E30A1D4A0…",1072,686,0.639925
"""020D35AA0D27D382678E2589894895…",1038,812,0.782274
"""E7C218002C3358D9BFFDB9D8C419E7…",876,558,0.636986
"""3D53A6FC25B80B680C205AFFF2AEFC…",817,138,0.168911
"""E48E44E522EDE22C361E1A0C6F2E68…",807,145,0.179678
"""89A28CB50E7F2F550BF66B50BF22F0…",739,515,0.696888
"""9E879AE06577D450ADBFAF9E110DE8…",728,535,0.73489
"""CDD75F662C8FC2F2985ACCF68CC263…",718,367,0.511142
